In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import csv

In [3]:
indiv_cols = [
    "CMTE_ID","AMNDT_IND","RPT_TP","TRANSACTION_PGI","IMAGE_NUM",
    "TRANSACTION_TP","ENTITY_TP","NAME","CITY","STATE","ZIP_CODE",
    "EMPLOYER","OCCUPATION","TRANSACTION_DT","TRANSACTION_AMT",
    "OTHER_ID","TRAN_ID","FILE_NUM","MEMO_CD","MEMO_TEXT",
    "SUB_ID"
]

cand_cols = [
    "CAND_ID","CAND_NAME","CAND_PTY_AFFILIATION","CAND_ELECTION_YR",
    "CAND_OFFICE_ST","CAND_OFFICE","CAND_OFFICE_DISTRICT","CAND_ICI",
    "CAND_STATUS","CAND_PCC","CAND_ST1","CAND_ST2","CAND_CITY",
    "CAND_ST","CAND_ZIP"
]

link_cols = [
    "CAND_ID","CAND_ELECTION_YR","FEC_ELECTION_YR",
    "CMTE_ID","CMTE_TP","CMTE_DSGN","LINKAGE_ID"
]

# Cleaning 2022 data

In [ ]:
chunk_size = 1000000

total_rows = 0
matched_rows = 0
first_write = True

reader = pd.read_csv(
    "../../data/itcon_2022.txt",
    sep="|",
    names=indiv_cols,
    dtype=str,
    chunksize=chunk_size,
    na_filter=False,
    quoting=csv.QUOTE_NONE,
    on_bad_lines="warn",
    engine="c",
    low_memory=True
)

for chunk in reader:
    total_rows += len(chunk)

    mask = (
        chunk["ENTITY_TP"].eq("IND") &
        chunk["TRANSACTION_PGI"].eq("G2022") &
        chunk["OTHER_ID"].eq("")
    )

    out = chunk.loc[mask]
    matched_rows += len(out)

    if not out.empty:
        out.to_csv(
            "../../data/indiv_pass1_2022.csv",
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )
        first_write = False

print(f"num of contributions: {total_rows}")
print(f"num matching filter: {matched_rows}")
print(f"saved to: ../../data/indiv_pass1_2022.csv")

num of contributions: 63885895
num matching filter: 2611570
saved to: ../../data/indiv_pass1_2022.csv


In [9]:
cand = pd.read_csv("../../data/cn_2022.txt", sep="|", names=cand_cols, dtype=str)
link = pd.read_csv("../../data/ccl_2022.txt", sep="|", names=link_cols, dtype=str)

print(f"num of candidates: {cand.shape[0]}")

num of candidates: 8580


In [12]:
# candidates running for senate in 2022 election
sen22 = cand[
    (cand["CAND_OFFICE"] == "S") &
    (cand["CAND_ELECTION_YR"] == "2022")
][["CAND_ID","CAND_NAME","CAND_PTY_AFFILIATION","CAND_OFFICE_ST"]].copy()

# link table for candidates committees in 2022 election
campaigns_of_interest = link[
    (link["CAND_ELECTION_YR"] == "2022") &
    (link["FEC_ELECTION_YR"] == "2022") &
    (link["CMTE_DSGN"].isin(["P","A"]))
]

sen22_cmtes = campaigns_of_interest.merge(sen22, on="CAND_ID", how="inner")

In [ ]:
# individual contributions and transaction in general 2022 election
indiv_pass1 = pd.read_csv("../../data/indiv_pass1_2022.csv", dtype=str)
indiv_pass1['TRANSACTION_DT'] = pd.to_datetime(indiv_pass1['TRANSACTION_DT'], 
                                               format='%m%d%Y', errors='coerce')
indiv_pass2 = indiv_pass1[
    (indiv_pass1["TRANSACTION_DT"].dt.year.isin([2021, 2022]))
]

# individual contributions to relevant campaigns, relevant data
senate_general_indiv = indiv_pass1.merge(
    sen22_cmtes[["CMTE_ID","CAND_ID","CAND_NAME","CAND_PTY_AFFILIATION","CAND_OFFICE_ST"]],
    on="CMTE_ID",
    how="inner"
)[['CMTE_ID', 'IMAGE_NUM', 'TRANSACTION_TP', 'NAME', 'CITY', 'STATE', 'ZIP_CODE', 
   'EMPLOYER', 'OCCUPATION', 'TRANSACTION_DT', 'TRANSACTION_AMT', 'CAND_ID', 'CAND_NAME',
   'CAND_PTY_AFFILIATION', 'CAND_OFFICE_ST']]

In [ ]:
# save cleaned data
senate_general_indiv.to_csv("../../data/senate_general_indiv22.csv", index=False)

In [15]:
senate_general_indiv.shape[0]

2017528

# Same for 2020 data

In [97]:
chunk_size = 1000000

total_rows = 0
matched_rows = 0
first_write = True

reader = pd.read_csv(
    "../../data/itcont_2020.txt",
    sep="|",
    names=indiv_cols,
    dtype=str,
    chunksize=chunk_size,
    na_filter=False,
    quoting=csv.QUOTE_NONE,
    on_bad_lines="warn",
    engine="c",
    low_memory=True
)

for chunk in reader:
    total_rows += len(chunk)

    mask = (
        chunk["ENTITY_TP"].eq("IND") &
        chunk["TRANSACTION_PGI"].eq("G2020") &
        chunk["OTHER_ID"].eq("")
    )

    out = chunk.loc[mask]
    matched_rows += len(out)

    if not out.empty:
        out.to_csv(
            "../../data/indiv_pass1_2020.csv",
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )
        first_write = False

print(f"num of contributions: {total_rows}")
print(f"num matching filter: {matched_rows}")
print(f"saved to: ../../data/indiv_pass1_2020.csv")

num of contributions: 69377425
num matching filter: 3686611
saved to: ../../data/indiv_pass1_2020.csv


In [98]:
cand = pd.read_csv("../../data/cn_2020.txt", sep="|", names=cand_cols, dtype=str)
link = pd.read_csv("../../data/ccl_2020.txt", sep="|", names=link_cols, dtype=str)

print(f"num of candidates: {cand.shape[0]}")

num of candidates: 7758


In [99]:
# candidates running for senate in 2020 election
sen20 = cand[
    (cand["CAND_OFFICE"] == "S") &
    (cand["CAND_ELECTION_YR"] == "2020")
][["CAND_ID","CAND_NAME","CAND_PTY_AFFILIATION","CAND_OFFICE_ST"]].copy()

# link table for candidates committees in 2020 election
campaigns_of_interest = link[
    (link["CAND_ELECTION_YR"] == "2020") &
    (link["FEC_ELECTION_YR"] == "2020") &
    (link["CMTE_DSGN"].isin(["P","A"]))
]

sen20_cmtes = campaigns_of_interest.merge(sen20, on="CAND_ID", how="inner")

In [100]:
# individual contributions and transaction in general 2020 election
indiv_pass1 = pd.read_csv("../../data/indiv_pass1_2020.csv", dtype=str)
indiv_pass1['TRANSACTION_DT'] = pd.to_datetime(indiv_pass1['TRANSACTION_DT'], 
                                               format='%m%d%Y', errors='coerce')
indiv_pass2 = indiv_pass1[
    (indiv_pass1["TRANSACTION_DT"].dt.year.isin([2019, 2020]))
]

# individual contributions to relevant campaigns, relevant data
senate_general_indiv = indiv_pass1.merge(
    sen20_cmtes[["CMTE_ID","CAND_ID","CAND_NAME","CAND_PTY_AFFILIATION","CAND_OFFICE_ST"]],
    on="CMTE_ID",
    how="inner"
)[['CMTE_ID', 'IMAGE_NUM', 'TRANSACTION_TP', 'NAME', 'CITY', 'STATE', 'ZIP_CODE', 
   'EMPLOYER', 'OCCUPATION', 'TRANSACTION_DT', 'TRANSACTION_AMT', 'CAND_ID', 'CAND_NAME',
   'CAND_PTY_AFFILIATION', 'CAND_OFFICE_ST']]

In [101]:
# save cleaned data
senate_general_indiv.to_csv("../../data/senate_general_indiv20.csv", index=False)

In [102]:
senate_general_indiv.shape[0]

2232708

## Race results

In [108]:
elections_res = pd.read_csv("../../data/election_results_senate.csv")

# 2020 and 2022 general election results only
elections_res = elections_res[
    (elections_res["cycle"].isin([2020, 2022])) &
    (elections_res["stage"] == "general")
].copy()

elections_res["percent"] = pd.to_numeric(elections_res["percent"], errors="coerce")
elections_res = elections_res.dropna(subset=["candidate_name", "percent"])

# final rounds for ranked-choice
def keep_final_round(group):
    if group["ranked_choice_round"].notna().any():
        max_round = group["ranked_choice_round"].max()
        return group[group["ranked_choice_round"] == max_round]
    return group

elections_res = (
    elections_res.groupby(["cycle", "state", "race_id"], group_keys=False)
      .apply(keep_final_round)
      .reset_index(drop=True)
)

In [113]:
elections_res = pd.read_csv("../../data/election_results_senate.csv")

# 2020 and 2022 general election results only
elections_res = elections_res[
    (elections_res["cycle"].isin([2020, 2022])) &
    (elections_res["stage"] == "general")
].copy()

elections_res["percent"] = pd.to_numeric(elections_res["percent"], errors="coerce")
elections_res = elections_res.dropna(subset=["candidate_name", "percent"])

# drop if ranked_choice_round is not NaN and is not last (3 for alaska)
mask = elections_res["ranked_choice_round"].isna() | (elections_res["ranked_choice_round"] == 3)
elections_res = elections_res[mask]

In [125]:
# keep top 2 candidates
elections_res = elections_res.sort_values(["cycle", "state_abbrev", "race_id", "percent"], 
                                          ascending=[True, True, True, False])
top2 = (
    elections_res.groupby(["cycle", "state_abbrev", "race_id"], group_keys=False)
      .head(2)
      .copy()
)
top2["rank"] = top2.groupby(["cycle", "state_abbrev", "race_id"]).cumcount() + 1

# compute margin
wide = top2.pivot(
    index=["cycle", "state_abbrev", "race_id"],
    columns="rank",
    values=["candidate_name", "percent", "votes"]
)
wide.columns = [f"{var}_{rank}" for var, rank in wide.columns]
wide = wide.reset_index()
wide = wide.dropna(subset=["candidate_name_1", "candidate_name_2", "percent_1", "percent_2"])

wide["margin_of_victory_pp"] = wide["percent_1"] - wide["percent_2"]
wide = wide.rename(columns={
    "cycle": "year",
    "candidate_name_1": "winner",
    "candidate_name_2": "runner_up",
    "percent_1": "winner_percent",
    "percent_2": "runner_up_percent",
    "votes_1": "winner_votes",
    "votes_2": "runner_up_votes"
})
result = wide[
    [
        "year",
        "state_abbrev",
        "race_id",
        "winner",
        "runner_up",
        "winner_percent",
        "runner_up_percent",
        "winner_votes",
        "runner_up_votes",
        "margin_of_victory_pp"
    ]
].sort_values(["year", "state_abbrev"])
result = result.rename(columns={"state_abbrev": "state"})

result.to_csv("../../data/senate_margin_of_victory_2020_2022.csv", index=False)